# RK1 Houseplant SFT — Colab T4

Стек: `transformers` + `TRL` + `PEFT` + `bitsandbytes`.

Перед запуском:
1. Runtime → GPU (T4).
2. Загрузите репозиторий или смонтируйте Drive с `data/processed/*.jsonl`.
3. Не коммитьте `OPENROUTER_API_KEY`; для judge задайте через Colab Secrets.

In [ ]:
# Optional: clone or mount
# from google.colab import drive; drive.mount('/content/drive')
REPO = '/content/LLM'  # change if needed
import os, sys
os.chdir(REPO) if os.path.isdir(REPO) else None
sys.path.insert(0, os.getcwd())
!nvidia-smi || true

In [ ]:
%pip install -q -U transformers trl peft bitsandbytes accelerate datasets pyyaml python-dotenv matplotlib pandas

## Pilot (10–20 steps): train → save → reload → 1 generation

In [ ]:
from scripts.train_qlora import main as _unused
import subprocess, sys
subprocess.check_call([sys.executable, 'scripts/train_qlora.py', '--pilot'])

## S1 full QLoRA (after pilot OK)

In [ ]:
# Uncomment for full run
# subprocess.check_call([sys.executable, 'scripts/train_qlora.py'])
print('Run full S1 after pilot succeeds')

## Eval helpers

После обучения сгенерируйте ответы base vs adapter одинаковыми decoding settings, затем:

```bash
python scripts/score_generations.py --mode general --predictions reports/metrics/general_dev_sft.jsonl --baseline reports/metrics/general_dev_base.jsonl
python scripts/score_generations.py --mode format --predictions reports/metrics/format_prompts_sft.jsonl
python scripts/run_judge.py --input reports/metrics/plant_dev_pair.jsonl --limit 20
```

Правило чекпоинта: plant_dev не хуже базы; general_dev drop ≤ 5 п.п.

In [ ]:
from src.eval.checkers import check_three_numbered_steps
demo_ok = '''1. Raise humidity.\n2. Trim brown tips.\n3. Check drainage.'''
demo_bad = '''1. Water less.\n2. Water less.\n3. Water less.'''
print(check_three_numbered_steps(demo_ok))
print(check_three_numbered_steps(demo_bad))